# Amazon ML Challenge 2026: Business Entity Resolution
## Phase 1 — Exploratory Data Analysis & Dataset Inspection
**Dataset Source:** `datasett11/student_resource/dataset`

This notebook provides a thorough, memory-efficient inspection of the challenge dataset:
- File sizes & row counts across all 7 TSV files
- Column schemas & data types
- Missing value & duplicate analysis
- Country distributions (US, India, France open-set behavior)
- Name and address length / token distributions
- Ground truth match behavior and singleton distribution
- Memory footprint & blocking strategy design

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Paths strictly pointing to datasett11
BASE_DIR = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else os.getcwd()
DATASET_ROOT = os.path.join(BASE_DIR, "datasett11", "student_resource", "dataset")
REPORT_PATH = os.path.join(BASE_DIR, "output", "datasett11_inspection_report.json")

print(f"Dataset root: {DATASET_ROOT}")

In [ ]:
# Load generated inspection report
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

# Summary Table of All 7 Files
summary_data = []
for name, data in report.items():
    if name == "train_ground_truth":
        summary_data.append({
            "File": name,
            "Rows": f"{data['total_s1_rows']:,}",
            "Size (MB)": f"{data['file_size_mb']:.2f}",
            "Unique IDs": f"{data['unique_s1_ids']:,}",
            "Duplicates": data['duplicate_s1_ids'],
            "Countries": "N/A (Ground Truth)",
            "Missing Addr (%)": "N/A",
        })
    else:
        summary_data.append({
            "File": name,
            "Rows": f"{data['total_rows']:,}",
            "Size (MB)": f"{data['file_size_mb']:.2f}",
            "Unique IDs": f"{data['unique_entity_ids']:,}",
            "Duplicates": data['duplicate_entity_ids'],
            "Countries": str(list(data['country_distribution'].keys())),
            "Missing Addr (%)": f"{data['missing_empty_percentages'].get('business_address', 0.0):.2f}%",
        })

df_summary = pd.DataFrame(summary_data)
print("=== DATASET OVERVIEW (datasett11) ===")
display(df_summary)

### Ground Truth Analysis
- S1 entities: singletons vs multiple matches
- Match links per Source 1 entity

In [ ]:
gt = report["train_ground_truth"]
print(f"Total Source 1 Entities: {gt['total_s1_rows']:,}")
print(f"Total Match Pairs (S1 -> S2/S3): {gt['total_matched_pairs']:,}")
print(f"Singletons (0 matches): {gt['zero_matches_count_singletons']:,} ({gt['zero_matches_pct_singletons']}%)")
print(f"1 match: {gt['one_match_count']:,} ({gt['one_match_pct']}%)")
print(f"2 matches: {gt['two_matches_count']:,} ({gt['two_matches_pct']}%)")
print(f"3+ matches: {gt['three_plus_matches_count']:,} ({gt['three_plus_matches_pct']}%)")
print(f"Max matches on a single S1: {gt['max_matches_per_s1']}")
print(f"Mean matches per S1: {gt['mean_matches_per_s1']:.4f}")

### Sample Inspection from Each Source

In [ ]:
for source_name in ["train_source1", "train_source2", "train_source3", "test_source1"]:
    print(f"\n--- Samples from {source_name} ---")
    samples = report[source_name]["sample_records"]
    display(pd.DataFrame(samples))